# Provider Split — NHS trusts vs independents

We separate the ~600 providers into **NHS trusts** (public, `R##` codes) and **independent** (private) hospitals, and from here on **use the trust-based ones only** for the federation — they're the stable, resourced clients (see `docs/nhs_structure.png`).

**Outputs:**
- `results/providers_classified.csv`, `results/trusts.csv`, `results/independents.csv`
- `data/processed/rtt_trusts_canonical.parquet` — the trust-only dataset downstream EDA uses.

Uses the loader in `src/rtt.py`. Kernel: **FedOPD (.venv)**.

## Cell 1 — setup

- `sys.path.insert(0, REPO)` — puts the repo root on Python's import path so `from src import rtt` works from this nested notebook folder.
- `from src import rtt` — our loader module (the canonical `[date, branch, dept, demand]` builder).
- `RESULTS` / `PROCESSED` — where we save the small provider lists (tracked in git) and the large trust-only dataset (under `data/`, git-ignored).
- `.mkdir(parents=True, exist_ok=True)` — create those folders if missing, no error if they already exist.

In [1]:
import sys
REPO = r"C:\Users\heave\Documents\FedOPD"
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd
from src import rtt

RESULTS = Path(REPO) / "notebooks" / "provider_split" / "results"
PROCESSED = Path(REPO) / "data" / "processed"
RESULTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
print("loader:", rtt.__file__)

loader: C:\Users\heave\Documents\FedOPD\src\rtt.py


## Cell 2 — build the canonical table for ALL providers

- `rtt.load_all(trusts_only=False)` — reads all 60 months and reduces each to canonical rows, keeping **everyone** (so we can measure the trust vs independent split). Internally it already: keeps `Part_3` (demand), drops the `C_999` Total specialty, and sums across commissioners.
- This reads ~5 GB of CSV, so it takes ~1–2 minutes the first time.
- `allp.shape` → expect ~211k rows × 8 columns.

In [2]:
allp = rtt.load_all(trusts_only=False)   # ~1-2 min
print("rows, cols:", allp.shape)
allp.head()

rows, cols: (211428, 8)


,date,branch,branch_name,icb,icb_name,dept,dept_name,demand
0,2024-06-01,8KL73,ENDOCARE DIAGNOSTICS,QOP,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,C_301,Gastroenterology Service,139
1,2024-07-01,8KL73,ENDOCARE DIAGNOSTICS,QOP,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,C_301,Gastroenterology Service,141
2,2024-08-01,8KL73,ENDOCARE DIAGNOSTICS,QOP,NHS GREATER MANCHESTER INTEGRATED CARE BOARD,C_301,Gastroenterology Service,122
3,2021-04-01,A0C5S,SPAMEDICA GLOUCESTER,QR1,GLOUCESTERSHIRE STP,C_130,Ophthalmology Service,118
4,2021-05-01,A0C5S,SPAMEDICA GLOUCESTER,QR1,GLOUCESTERSHIRE STP,C_130,Ophthalmology Service,125


## Cell 3 — collapse to one row per provider + classify

- `allp.groupby("branch", ...)` — group all rows by provider code, then aggregate.
- `.agg(total_demand=("demand", "sum"), months_present=("date", "nunique"), ...)` — per provider: total lifetime demand, how many of the 60 months it appears in, plus its name/ICB (`"first"` just takes one value since they're constant per provider).
- `prov["is_trust"] = prov["branch"].map(rtt.is_nhs_trust)` — flag each provider `True` if it's an NHS trust (`R##`).
- `.sort_values("total_demand", ascending=False)` — biggest providers first.

In [3]:
prov = allp.groupby("branch", as_index=False).agg(
    branch_name=("branch_name", "first"),
    icb=("icb", "first"),
    icb_name=("icb_name", "first"),
    total_demand=("demand", "sum"),
    months_present=("date", "nunique"),
)
prov["is_trust"] = prov["branch"].map(rtt.is_nhs_trust)
prov = prov.sort_values("total_demand", ascending=False)
prov.head(10)

,branch,branch_name,icb,icb_name,total_demand,months_present,is_trust
408,RAJ,MID AND SOUTH ESSEX NHS FOUNDATION TRUST,QH8,MID AND SOUTH ESSEX STP,2404591,60,True
387,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,QOP,GREATER MANCHESTER HEALTH & SOCIAL CARE PARTNE...,1748756,57,True
509,RTD,THE NEWCASTLE UPON TYNE HOSPITALS NHS FOUNDATI...,QHM,CUMBRIA AND NORTH EAST STP,1703762,60,True
455,RJ1,GUY'S AND ST THOMAS' NHS FOUNDATION TRUST,QKK,OUR HEALTHIER SOUTH EAST LONDON STP,1644406,60,True
506,RRK,UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION...,QHL,BIRMINGHAM AND SOLIHULL STP,1489398,60,True
465,RJZ,KING'S COLLEGE HOSPITAL NHS FOUNDATION TRUST,QKK,OUR HEALTHIER SOUTH EAST LONDON STP,1415414,60,True
502,RR8,LEEDS TEACHING HOSPITALS NHS TRUST,QWO,WEST YORKSHIRE AND HARROGATE HEALTH & CARE PAR...,1413166,60,True
396,R1H,BARTS HEALTH NHS TRUST,QMF,EAST LONDON HEALTH & CARE PARTNERSHIP (STP),1408489,60,True
476,RM3,SALFORD ROYAL NHS FOUNDATION TRUST,QOP,GREATER MANCHESTER HEALTH & SOCIAL CARE PARTNE...,1392180,60,True
559,RYJ,IMPERIAL COLLEGE HEALTHCARE NHS TRUST,QRV,NORTH WEST LONDON HEALTH & CARE PARTNERSHIP (STP),1339402,60,True


## Cell 4 — how big is each group?

- Split `prov` into trusts and independents with a boolean mask (`~` = NOT).
- Compare **counts** and **share of total demand** — this is the evidence for using trusts only.

In [4]:
trusts = prov[prov["is_trust"]]
indep  = prov[~prov["is_trust"]]
tot = prov["total_demand"].sum()

print(f"providers total : {len(prov)}")
print(f"  NHS trusts    : {len(trusts):>4}  ({trusts['total_demand'].sum()/tot:.1%} of demand)")
print(f"  independents  : {len(indep):>4}  ({indep['total_demand'].sum()/tot:.1%} of demand)")

providers total : 598
  NHS trusts    :  172  (92.2% of demand)
  independents  :  426  (7.8% of demand)


## Cell 5 — save the split + the trust-only dataset

- Provider **lists** → `results/` (small CSVs, committed to git as a reference).
- The **trust-only canonical table** → `data/processed/rtt_trusts_canonical.parquet` (larger; `data/` is git-ignored, and parquet is a fast compressed columnar format).
- Downstream EDA loads that parquet instead of re-reading 60 CSVs each time.

In [5]:
prov.to_csv(RESULTS / "providers_classified.csv", index=False)
trusts.to_csv(RESULTS / "trusts.csv", index=False)
indep.to_csv(RESULTS / "independents.csv", index=False)

trust_df = allp[allp["branch"].map(rtt.is_nhs_trust)].reset_index(drop=True)
trust_df.to_parquet(PROCESSED / "rtt_trusts_canonical.parquet", index=False)
print("trust-only canonical:", trust_df.shape, "->", PROCESSED / "rtt_trusts_canonical.parquet")

trust-only canonical: (147986, 8) -> C:\Users\heave\Documents\FedOPD\data\processed\rtt_trusts_canonical.parquet


## Result & decision

Expected numbers:

| group | providers | share of demand |
|-------|-----------|-----------------|
| **NHS trusts** | **172** | **92.2%** |
| independents | 426 | 7.8% |

→ **Decision:** proceed with the **172 NHS trusts** — they carry 92% of demand and are stable, directly-governed clients. The independents (many tiny private clinics) are set aside for the federation.

Trust-only dataset: **147,986 rows · 60 months · 172 trusts · 23 departments**, saved to `data/processed/rtt_trusts_canonical.parquet`. All downstream EDA uses this file.

→ **Next:** deeper EDA (Step 4) on the trust-only data — balanced panel (trusts present all 60 months), scale per trust/dept, trusts-per-ICB.